# **Data Preprocessing** 

Notebook này chạy toàn bộ quy trình tiền xử lý dữ liệu (Data Preprocessing) để chuẩn bị dữ liệu cho modeling.

**Các bước thực hiện:**
- **Bước 1.** Data Cleaning:
   - Xử lý Missing Value dạng numeric
   - Xử lý Missing Value dạng text
   - Xử lý Outliers
   - Làm sạch text

- **Bước 2.** Data Reduction:
   - Drop các cột không cần thiết

- **Bước 3.** Data Transformation:
   - Feature Engineering (tạo 11 features mới)
   - One-hot encoding cho categorical variables

- **Bước 4.** Scaling:
   - Chia tập train, text
   - MinMaxScaler 


### **Import các thư viện cần thiết và setup đường dẫn**
### **Import thư viện và setup đường dẫn**
Thêm đường dẫn src vào PYTHONPATH để import các module tự định nghĩa
- `os.getcwd()`: Lấy thư mục hiện tại
- Kiểm tra nếu đang ở trong thư mục `notebooks/`, lên 1 cấp để về project root
- `sys.path.insert(0, src_path)`: Thêm đường dẫn src/ vào đầu Python path để Python có thể tìm thấy các module trong thư mục `src/`

In [1]:
# Import các thư viện cần thiết
import sys
import os
import pandas as pd

# Thêm đường dẫn src vào PYTHONPATH để import các module
# Lấy thư mục cha của notebooks/ (tức là project root)
current_dir = os.getcwd()
# Nếu đang ở trong thư mục notebooks/, lên 1 cấp
if current_dir.endswith('notebooks'):
    project_root = os.path.dirname(current_dir)
else:
    project_root = current_dir

src_path = os.path.join(project_root, 'src')
sys.path.insert(0, src_path)

print("Đã import thành công các module cần thiết!")

Đã import thành công các module cần thiết!


### **Đọc dữ liệu đã được xử lý kiểu dữ liệu**

Trước khi chạy preprocessing, cần đọc file `data_typed.csv` đã được xử lý kiểu dữ liệu từ bước Data Exploration.


In [2]:
# Đọc file data_typed.csv đã được xử lý kiểu dữ liệu
df = pd.read_csv("../data/processed/data_typed.csv")

print(f"Đã đọc file thành công")
print(f"Dataframe {df.shape}")
print(f"\n5 dòng đầu tiên:")
display(df.head())

Đã đọc file thành công
Dataframe (19811, 29)

5 dòng đầu tiên:


,product_id,product_name,product_url,category_id,category_name,category_root_name,price,original_price,discount_rate,quantity_sold,...,origin,store_id,store_name,store_review_count,total_follower,is_official,cancel_by_seller_rate,cancel_by_seller_rate_status,return_rate,return_rate_status
0,276641167,Xe máy Honda Vision 2025 Cao Cấp,https://tiki.vn/xe-may-honda-vision-2025-cao-c...,8597,Xe máy,Ô Tô – Xe Máy – Xe Đạp,34500000.0,34600000.0,0.0,21.0,...,Nhật Bản / Việt Nam,18792.0,HEAD Phát Tiến,4767.0,5517.0,0.0,0.0,normal,0.0,normal
1,276626995,Xe Máy Honda Vision 2025 - Phiên Bản Thể Thao,https://tiki.vn/xe-may-honda-vision-2025-phien...,8597,Xe máy,Ô Tô – Xe Máy – Xe Đạp,38659000.0,38759000.0,0.0,32.0,...,Việt Nam,180553.0,Honda ủy nhiệm Hoàng Việt,919.0,2265.0,0.0,0.0,normal,0.0,normal
2,276626834,Xe Máy Honda Vision 2025 - Phiên Bản Đặc Biệt,https://tiki.vn/xe-may-honda-vision-2025-phien...,8597,Xe máy,Ô Tô – Xe Máy – Xe Đạp,36608000.0,36708000.0,0.0,25.0,...,Việt Nam,180553.0,Honda ủy nhiệm Hoàng Việt,919.0,2265.0,0.0,0.0,normal,0.0,normal
3,276626812,Xe máy Honda Vision 2025 Đặc Biệt,https://tiki.vn/xe-may-honda-vision-2025-dac-b...,8597,Xe máy,Ô Tô – Xe Máy – Xe Đạp,36920000.0,37020000.0,0.0,15.0,...,Nhật Bản / Việt Nam,18792.0,HEAD Phát Tiến,4767.0,5517.0,0.0,0.0,normal,0.0,normal
4,276626642,Xe Máy Honda Vision 2025 - Phiên Bản Cao Cấp,https://tiki.vn/xe-may-honda-vision-2025-phien...,8597,Xe máy,Ô Tô – Xe Máy – Xe Đạp,34762000.0,34862000.0,0.0,16.0,...,Việt Nam,180553.0,Honda ủy nhiệm Hoàng Việt,919.0,2265.0,0.0,0.0,normal,0.0,normal


## **Import các hàm trong `data_preprocessing.py`**

Chức năng của các hàm import:
- `handle_numeric_missing_values`: Xử lý NaN trong cột số
- `handle_text_missing_values`: Xử lý NaN trong cột text
- `handle_outliers`: Xử lý outliers (Winsorizing + Log)
- `clean_text_columns`: Chuẩn hóa text (normalize origin)
- `reduce_data`: Drop các cột không cần thiết
- `create_features`: Tạo 11 features mới
- `encode_categorical`: One-hot encoding cho category & origin
- `scale_and_split`: chia tập train/test và scaling

In [3]:
# Import các hàm cần thiết từ module
from data_preprocessing import (
    handle_numeric_missing_values, handle_text_missing_values,
    handle_outliers, clean_text_columns, reduce_data,
    create_features, encode_categorical, scale_and_split
)

print("Đã import tất cả các hàm cần thiết!")

Đã import tất cả các hàm cần thiết!


## **Bước 1. Data Cleaning**

### **Xử lý Missing Value dạng numeric**

Xử lý missing values cho các cột số:
- **Fill 0.0**: cho các cột`quantity_sold`, `is_*`, `store_id`
- **KNN Imputer**: cho các cột còn lại `store_review_count`, `total_follower`, `is_official`, `cancel_by_seller_rate`, `return_rate`

KNN Imputer sử dụng 5 neighbors với weights='distance' để điền giá trị missing dựa trên các điểm dữ liệu gần nhất.


In [4]:
print("="*60)
print("XỬ LÝ MISSING VALUE NUMERIC")
print("="*60)

df = handle_numeric_missing_values(df)

print(f"\Dataframe sau khi xử lý missing numeric: {df.shape}")
print(f"Số missing values còn lại: {df.isnull().sum().sum()}")

XỬ LÝ MISSING VALUE NUMERIC
Đã điền 0.0 cho quantity_sold: 5639 giá trị
Đã điền 0.0 cho is_freeship_xtra: 63 giá trị
Đã điền 0.0 cho is_authentic: 63 giá trị
Đã điền 0.0 cho is_official: 63 giá trị
Đã điền 0.0 cho store_id: 114 giá trị
Các cột sẽ dùng KNN Imputer (4 cột):
1. store_review_count: 114 missing
2. total_follower: 114 missing
3. cancel_by_seller_rate: 114 missing
4. return_rate: 114 missing
\Dataframe sau khi xử lý missing numeric: (19811, 29)
Số missing values còn lại: 1667


<>:7: SyntaxWarning: invalid escape sequence '\D'
<>:7: SyntaxWarning: invalid escape sequence '\D'
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_15276\3738396306.py:7: SyntaxWarning: invalid escape sequence '\D'
  print(f"\Dataframe sau khi xử lý missing numeric: {df.shape}")


### **Xử lý Missing Value dạng text**

Xử lý missing values cho các cột text:
- `brand_name`, origin: Điền "Unknown"
- `store_name`: Mapping từ `store_id` nếu tìm thấy trong dataset, nếu không thì "Unknown"
- `cancel_by_seller_rate_status`: Điền mode nếu có `cancel_by_seller_rate`, "Unknown" nếu không
- `return_rate_status`: Điền mode nếu có `return_rate`, "Unknown" nếu không


In [5]:
print("\n" + "="*60)
print("XỬ LÝ MISSING VALUE TEXT")
print("="*60)

df = handle_text_missing_values(df)

print(f"\Dataframe sau khi xử lý missing text: {df.shape}")
print(f"Số missing values còn lại: {df.isnull().sum().sum()}")


XỬ LÝ MISSING VALUE TEXT


<>:7: SyntaxWarning: invalid escape sequence '\D'
<>:7: SyntaxWarning: invalid escape sequence '\D'
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_15276\1022260412.py:7: SyntaxWarning: invalid escape sequence '\D'
  print(f"\Dataframe sau khi xử lý missing text: {df.shape}")


\Dataframe sau khi xử lý missing text: (19811, 29)
Số missing values còn lại: 0


In [6]:
df.to_csv("../data/processed/fill_missing_values_data.csv")

### **Xử lý Outliers**

Xử lý outliers bằng 2 phương pháp:

1. **Winsorizing + Log Transformation** cho:
   - `price`, `original_price`, `quantity_sold`
   - `review_count`, `video_count`, `store_review_count`, `total_follower`

2. **Chỉ Winsorizing** cho:
   - `discount_rate`, `image_count`


In [7]:
# 2.1.3: Xử lý Outliers
print("\n" + "="*60)
print("XỬ LÝ OUTLIERS")
print("="*60)

df = handle_outliers(df)

print(f"\nShape sau khi xử lý outliers: {df.shape}")
print("\nĐã áp dụng Winsorizing và Log Transformation cho các cột số lớn")



XỬ LÝ OUTLIERS

Shape sau khi xử lý outliers: (19811, 29)

Đã áp dụng Winsorizing và Log Transformation cho các cột số lớn


### **Làm sạch text**

Chuẩn hóa các cột text:
- `origin`: Chuẩn hóa tên quốc gia về tiếng Anh (Vietnam, China, Japan, etc.)
- Xử lý các dạng viết khác nhau (Việt Nam, VietNam, VN -> Vietnam)
- Tách và xử lý các giá trị có nhiều quốc gia (phân cách bởi dấu phẩy, '/', '-')

In [8]:
print("\n" + "="*60)
print("LÀM SẠCH TEXT")
print("="*60)

df = clean_text_columns(df)

print(f"\nShape sau khi làm sạch text: {df.shape}")
if 'origin' in df.columns:
    print(f"\nCác giá trị origin unique: {df['origin'].nunique()}")
    print(f"Ví dụ các giá trị origin: {df['origin'].unique()[:10]}")

df.to_csv(f'../data/processed/data_before_reduction.csv', index=False)


LÀM SẠCH TEXT

Shape sau khi làm sạch text: (19811, 29)

Các giá trị origin unique: 96
Ví dụ các giá trị origin: ['Japan, Vietnam' 'Vietnam' 'Other' 'Taiwan' 'Japan' 'China' 'Hong Kong'
 'China, Hong Kong' 'Taiwan, Vietnam' 'China, Vietnam']


## **Bước 2. Data Reduction**

Drop các cột không cần thiết:
- **Numeric**: `return_rate`, `cancel_by_seller_rate` (không có giá trị dự đoán)
- **Text**: `cancel_by_seller_rate_status`, `return_rate_status`, `product_url`, `category_name`, `store_name`, `brand_name` (chỉ giữ `category_root_name`)
- **ID**: `product_id`, `store_id`, `category_id` (chỉ dùng để định danh)


In [9]:
print("\n" + "="*60)
print("DATA REDUCTION")
print("="*60)

print(f"Số cột trước khi reduction: {len(df.columns)}")
df = reduce_data(df)
print(f"Số cột sau khi reduction: {len(df.columns)}")
print(f"Dataframe sau reduction: {df.shape}")


DATA REDUCTION
Số cột trước khi reduction: 29
Số cột sau khi reduction: 18
Dataframe sau reduction: (19811, 18)


## **Bước 3: Data Transformation**

### **Feature Engineering**

Tạo 11 features mới:

- `reputation_score`: Điểm uy tín thực sự
- `trust_level`: Điểm tín nhiệm
- `review_to_sold_ratio`: Tỷ lệ tương tác
- `shop_potential`: Độ nổi tiếng shop
- `total_visuals`: Tổng tư liệu hình ảnh và video
- `has_video`: Flag có video không (0/1)
- `discount_amount`: Số tiền giảm thực tế
- `price_vs_category`: Giá tương đối so với danh mục
- `hot_keyword_count`: Đếm từ khóa "giật tít" trong `product_name`
- `name_length`: Độ dài tiêu đề (ký tự)
- `name_word_count`: Số từ trong tiêu đề

Sau khi tạo features, cột `product_name` sẽ bị drop.


In [10]:
print("\n" + "="*60)
print("FEATURE ENGINEERING")
print("="*60)

print(f"Số cột trước khi tạo features: {len(df.columns)}")
df = create_features(df)
print(f"Số cột sau khi tạo features: {len(df.columns)}")
print(f"\Dataframe sau feature engineering: {df.shape}")

# Hiển thị các features mới
new_features = ['reputation_score', 'trust_level', 'review_to_sold_ratio', 
                'shop_potential', 'total_visuals', 'has_video', 
                'discount_amount', 'price_vs_category', 'hot_keyword_count',
                'name_length', 'name_word_count']
existing_new_features = [f for f in new_features if f in df.columns]
print(f"\nCác features mới đã tạo: {existing_new_features}")


FEATURE ENGINEERING
Số cột trước khi tạo features: 18


<>:8: SyntaxWarning: invalid escape sequence '\D'
<>:8: SyntaxWarning: invalid escape sequence '\D'
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_15276\47257366.py:8: SyntaxWarning: invalid escape sequence '\D'
  print(f"\Dataframe sau feature engineering: {df.shape}")


Số cột sau khi tạo features: 40
\Dataframe sau feature engineering: (19811, 40)

Các features mới đã tạo: ['reputation_score', 'trust_level', 'review_to_sold_ratio', 'shop_potential', 'total_visuals', 'has_video', 'discount_amount', 'price_vs_category', 'hot_keyword_count', 'name_length', 'name_word_count']


### **Chuẩn hóa dữ liệu (One-hot encoding)**

Áp dụng One-hot encoding cho các biến:

- `category_root_name`: 
   - One-hot encoding với `drop_first=True` (tránh multicollinearity)
   - Tạo các cột dạng `category_root_name_Category1`, `category_root_name_Category2`, ...

- `origin`:
   - MultiLabel Binarizer (vì một sản phẩm có thể có nhiều quốc gia)
   - Tách các giá trị phân cách bởi dấu phẩy, '/', '-'
   - Gộp các cột có tỉ lệ <= 2% vào cột "Others"


In [11]:
print("\n" + "="*60)
print("CHUẨN HÓA DỮ LIỆU (ONE-HOT ENCODING)")
print("="*60)

print(f"Số cột trước khi encoding: {len(df.columns)}")
df_processed = encode_categorical(df)
print(f"Số cột sau khi encoding: {len(df_processed.columns)}")
print(f"\nShape sau encoding: {df_processed.shape}")
df_processed.to_csv(f'../data/processed/data_processed.csv', index=False)

print("\n" + "="*60)
print("PREPROCESSING HOÀN TẤT!")
print("="*60)


CHUẨN HÓA DỮ LIỆU (ONE-HOT ENCODING)
Số cột trước khi encoding: 40
One-hot encoding cho category_root_name hoàn tất
Số cột sau khi encoding: 67

Shape sau encoding: (19811, 67)

PREPROCESSING HOÀN TẤT!


## **Bước 4: Chia Train-Test và Scaling**

Sau khi preprocessing, cần:
1. **Chia train-test split**: 80% train, 20% test
2. **Áp dụng MinMaxScaler**: Scale các cột numeric về khoảng [0, 1]
   - Không scale các cột binary (is_official, is_authentic, etc.)
   - Xử lý NaN và inf trước khi scale

- Lưu 3 file:
  - `train_data_final.csv`: Dữ liệu train đã scale
  - `test_data_final.csv`: Dữ liệu test đã scale
  - `full_data_final.csv`: Toàn bộ dữ liệu đã preprocessing (chưa scale)


In [12]:
train_df, test_df, scaler = scale_and_split(
    df_processed, 
    test_size=0.2, 
    random_state=42,
    save_files=True
)

print("\n" + "="*60)
print("SCALING HOÀN TẤT!")
print("="*60)
print(f"\nTrain dataframe: {train_df.shape}")
print(f"Test dataframe: {test_df.shape}")
print(f"\nCác file đã được lưu:")
print("- ../data/processed/train_data_final.csv")
print("- ../data/processed/test_data_final.csv")
print("- ../data/processed/full_data_final.csv")



SCALING HOÀN TẤT!

Train dataframe: (15848, 67)
Test dataframe: (3963, 67)

Các file đã được lưu:
- ../data/processed/train_data_final.csv
- ../data/processed/test_data_final.csv
- ../data/processed/full_data_final.csv


## Xem kết quả

Kiểm tra dữ liệu sau khi preprocessing và scaling:


In [13]:
# Xem thông tin về train set
print("THÔNG TIN TRAIN SET:")
print("="*60)
print(f"Shape: {train_df.shape}")
print(f"\nCác cột trong train set:")
print(train_df.columns.tolist())
print(f"\n5 dòng đầu tiên:")
display(train_df.head())


THÔNG TIN TRAIN SET:
Shape: (15848, 67)

Các cột trong train set:
['price', 'original_price', 'discount_rate', 'quantity_sold', 'rating_average', 'review_count', 'is_return_policy', 'is_freeship_xtra', 'is_authentic', 'image_count', 'video_count', 'is_brand', 'store_review_count', 'total_follower', 'is_official', 'reputation_score', 'trust_level', 'review_to_sold_ratio', 'shop_potential', 'total_visuals', 'has_video', 'discount_amount', 'price_vs_category', 'kw_chính hãng', 'kw_cao cấp', 'kw_freeship', 'kw_xịn', 'kw_hot', 'kw_giảm', 'kw_sale', 'kw_tặng', 'kw_combo', 'kw_bảo hành', 'kw_nhập khẩu', 'kw_siêu rẻ', 'hot_keyword_count', 'name_length', 'name_word_count', 'category_root_name_Bách Hóa Online', 'category_root_name_Chăm sóc nhà cửa', 'category_root_name_Giày - Dép nam', 'category_root_name_Giày - Dép nữ', 'category_root_name_Laptop – Máy Vi Tính – Linh kiện', 'category_root_name_Làm Đẹp - Sức Khỏe', 'category_root_name_Máy Ảnh - Máy Quay Phim', 'category_root_name_NGON', 'categor

,price,original_price,discount_rate,quantity_sold,rating_average,review_count,is_return_policy,is_freeship_xtra,is_authentic,image_count,...,category_root_name_Điện Thoại - Máy Tính Bảng,category_root_name_Điện Tử - Điện Lạnh,category_root_name_Đồ chơi - Mẹ & Bé,China,Japan,South Korea,Thailand,USA,Vietnam,Others
17463,0.445108,0.438575,0.000000,0.417519,0.74,0.331717,1.0,1.0,1.0,0.411765,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.00
14152,0.313109,0.343930,0.500000,0.417519,1.00,0.367314,1.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25
7346,0.156620,0.148556,0.000000,0.000000,0.00,0.000000,0.0,1.0,1.0,0.176471,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.00
3048,0.263237,0.330742,0.846154,0.351023,0.86,0.310416,1.0,1.0,1.0,0.294118,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25
2252,0.396891,0.390102,0.000000,0.212204,0.80,0.175253,1.0,1.0,1.0,0.235294,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.00


In [14]:
# Xem thông tin về test set
print("THÔNG TIN TEST SET:")
print("="*60)
print(f"Shape: {test_df.shape}")
print(f"\n5 dòng đầu tiên:")
display(test_df.head())


THÔNG TIN TEST SET:
Shape: (3963, 67)

5 dòng đầu tiên:


,price,original_price,discount_rate,quantity_sold,rating_average,review_count,is_return_policy,is_freeship_xtra,is_authentic,image_count,...,category_root_name_Điện Thoại - Máy Tính Bảng,category_root_name_Điện Tử - Điện Lạnh,category_root_name_Đồ chơi - Mẹ & Bé,China,Japan,South Korea,Thailand,USA,Vietnam,Others
6180,0.069133,0.060604,0.000000,0.351023,1.0,0.175253,1.0,1.0,1.0,0.117647,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.00
5605,0.470614,0.464217,0.000000,0.000000,0.0,0.000000,1.0,1.0,1.0,0.411765,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.00
15667,0.000000,0.000000,0.000000,0.195394,0.0,0.000000,1.0,1.0,1.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.00
9123,0.443963,0.444183,0.096154,0.377944,0.9,0.382518,1.0,1.0,1.0,0.588235,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25
2796,0.659291,0.653896,0.000000,0.000000,0.0,0.000000,1.0,1.0,1.0,0.352941,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.00
